# Segmentación de las 961 fotos del proyecto (J0, J1, M-b3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/datasantana/sam-image-processing/blob/main/notebooks/predict_all.ipynb)

Este notebook **solo orquesta**; la lógica está en `segpipe/predict.py` (probada con `pytest` en CPU).

- **Ensamble** (las 961 fotos): media de las probabilidades softmax de los 5 modelos de pliegue y argmax. Cada modelo infiere exactamente como en su evaluación (J0: predice a 512 y escala los logits; J1 y M-b3: ventana deslizante con el solape y la escala de su config).
- **Fuera de pliegue** (`_oof`, solo las 66 etiquetadas): el modelo del pliegue donde la foto fue de test. Se usa para verificar contra la CV (|ΔmIoU| ≤ 0.1 pp; si no, se detiene).
- El modelo predice siempre una de las 12 clases (no hay "sin anotar"): cielo y fondo lejano reciben alguna. No se corrige; se reporta por foto la fracción de píxeles con probabilidad máxima < 0.5, < 0.7 y < 0.9.

Salidas en `MyDrive/segpipe/pascual_v1/predictions/<config>/`. **Reanudable**: si la sesión se corta, vuelve a ejecutar las mismas celdas; se saltan las fotos ya escritas (`progress.jsonl`).

**Usa una A100** (en T4 hay un fallo de cuDNN sin resolver): *Entorno de ejecución → Cambiar tipo → A100*.

## 1. Código, dependencias y GPU

In [ ]:
REPO_URL = "https://github.com/datasantana/sam-image-processing.git"
BRANCH = "main"
REPO_DIR = "/content/sam-image-processing"

import os, subprocess, sys
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# Usa el torch que trae Colab; instala solo lo que falte.
import importlib.util
need = {"segmentation_models_pytorch": "segmentation-models-pytorch", "transformers": "transformers",
        "albumentations": "albumentations", "timm": "timm", "yaml": "pyyaml", "cv2": "opencv-python-headless"}
missing = [pkg for mod, pkg in need.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
from segpipe.env import gpu_info, versions
gpu = gpu_info()
print(versions()); print(gpu)
if not gpu["name"] or "A100" not in gpu["name"]:
    raise SystemExit(f"Se necesita una A100 (GPU actual: {gpu['name']}). En T4 hay un fallo de cuDNN sin resolver.")

## 2. Parámetros y dataset etiquetado (máscaras de CVAT para la verificación)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

STUDY_DIR = "/content/drive/MyDrive/segpipe/pascual_v1"        # corridas de la CV (solo lectura)
PHOTOS_DRIVE = "/content/drive/MyDrive/segpipe/fotos_pascual"   # 961 JPG, directamente en la carpeta
OUT_ROOT = STUDY_DIR + "/predictions"                           # salidas (Drive)
LOCAL_PHOTOS = "/content/fotos_pascual"                         # copia local en la VM
STAGING = "/content/predict_staging"                            # temporales locales
SMOKE_OUT = "/content/predictions_smoke"                        # humo: solo local
CONFIGS = ["J0", "J1", "M-b3"]
SEED = 42
DATASET_CFG = "configs/datasets/pascual.yaml"

In [ ]:
import json
import numpy as np
from pathlib import Path
from PIL import Image
from segpipe.data_io import load_dataset_config, ensure_dataset, read_manifest
from segpipe.classes import ClassSchema
from segpipe import predict as P

dcfg = load_dataset_config(DATASET_CFG)
root = ensure_dataset(dcfg, verbose=True)             # copia local validada del segpack (sha256)
lay, ccfg = dcfg["layout"], dcfg.get("classes", {}) or {}
schema = ClassSchema.from_json(root / lay["classes"], ignore=ccfg.get("ignore"),
                               mapping=ccfg.get("mapping"), target=ccfg.get("target"))
rows = read_manifest(root, lay["manifest"])
study_classes = json.loads(Path(STUDY_DIR, "study.json").read_text(encoding="utf-8"))["params"]["classes"]
assert schema.names == study_classes, (schema.names, study_classes)
by_name = {r["name"]: r for r in rows}

def mask_index(name):     # máscara de CVAT → índices de entrenamiento (255 = sin anotar)
    with Image.open(root / by_name[name]["mask"]) as im:
        return schema.rgb_to_index(np.asarray(im.convert("RGB")), where=name)

def mask_rgb(name):       # máscara de CVAT tal cual (para la hoja de contacto)
    with Image.open(root / by_name[name]["mask"]) as im:
        return np.asarray(im.convert("RGB"))

print(len(rows), "fotos etiquetadas;", schema.num_classes, "clases:", schema.names)

## 3. Verificación inicial y copia local

Primero sobre Drive (conteo = 961 y las 66 etiquetadas presentes); luego se copian las fotos al disco de la VM y se repite completa sobre la copia: tamaño 4032×2268, orientación EXIF = 1, sha256 de las 66 igual al del manifest, y la misma asignación foto → pliegue de test en J0, J1 y M-b3. **Si algo no cuadra, se detiene y lo lista todo.**

In [ ]:
splits = {c: P.load_splits(STUDY_DIR, c, SEED) for c in CONFIGS}
P.preflight(PHOTOS_DRIVE, rows, splits, full=False)
photos = P.stage_photos(P.list_photos(PHOTOS_DRIVE), LOCAL_PHOTOS)
pre = P.preflight(LOCAL_PHOTOS, rows, splits, full=True)
labelled = pre["labelled_fold"]
print("etiquetadas por pliegue:", {f: sum(v == f for v in labelled.values()) for f in sorted(set(labelled.values()))})

## 4. Corrida de humo (5 fotos) y estimación de tiempo

2 etiquetadas (ambas versiones) y 3 sin etiquetar, por config, en `SMOKE_OUT` (local, no se sube a Drive). Mide segundos de inferencia y de escritura por foto y estima las horas de la corrida completa (961 del ensamble + 66 fuera de pliegue por config). **Revisa la estimación y pon `CONFIRMAR = True` antes de la sección 5.**

In [ ]:
import shutil, torch
device = torch.device("cuda")
amp = P.amp_dtype_for(device)
palette = P.palette_from_schema(schema)
lab_names = sorted(labelled)[:2]
unl_names = sorted(set(photos) - set(labelled))[:3]
smoke_photos = {n: photos[n] for n in lab_names + unl_names}
shutil.rmtree(SMOKE_OUT, ignore_errors=True)
est = {}
for c in CONFIGS:
    members = P.load_members(STUDY_DIR, c, schema.names, device, SEED)
    r = P.predict_config(c, members, smoke_photos, {n: labelled[n] for n in lab_names}, f"{SMOKE_OUT}/{c}",
                         palette, schema.names, device, amp, ("oof", "ensamble"), mask_fn=mask_index,
                         staging=STAGING)
    est[c] = P.estimate_hours(r, n_ensemble=len(photos), n_oof=len(labelled))
    del members; torch.cuda.empty_cache()
for c, e in est.items():
    pv = e["por_version"]
    print(f"{c}: ensamble {pv['ensamble']['s_inferencia']:.1f} s/foto (escritura {pv['ensamble']['s_escritura']:.1f} s), "
          f"oof {pv['oof']['s_inferencia']:.1f} s/foto → ~{e['horas']:.2f} h")
print(f"TOTAL estimado: ~{sum(e['horas'] for e in est.values()):.1f} h de A100")

## 5. Corrida completa (reanudable)

In [ ]:
CONFIRMAR = False      # ← True después de revisar la estimación de la sección 4
if not CONFIRMAR:
    raise SystemExit("Revisa la estimación y pon CONFIRMAR = True.")
results = {}
for c in CONFIGS:
    members = P.load_members(STUDY_DIR, c, schema.names, device, SEED)
    results[c] = P.run_config(c, STUDY_DIR, photos, labelled, OUT_ROOT, schema, device, mask_fn=mask_index,
                              seed=SEED, staging=STAGING, members=members)   # se detiene si la verificación falla
    del members; torch.cuda.empty_cache()

## 6. Verificación fuera de pliegue y confianza baja

In [ ]:
import pandas as pd
ver = []
for c in CONFIGS:
    v = json.loads(Path(OUT_ROOT, c, "verificacion_oof.json").read_text(encoding="utf-8"))
    ver.append({"config": c, "mIoU oof (%)": round(100 * v["mIoU_oof"], 2), "mIoU CV (%)": round(100 * v["mIoU_cv"], 2),
                "Δ (pp)": round(v["delta_pp"], 3), "pasa (≤ 0.1 pp)": v["passed"],
                "px distintos máx (cota inf.)": v["px_distintos_max"], "foto": v["foto_px_distintos_max"],
                "fotos con diferencia": f"{v['fotos_con_diferencia']}/{v['n_fotos']}"})
display(pd.DataFrame(ver))
low = []
for c in CONFIGS:
    m = pd.read_csv(Path(OUT_ROOT, c, "predictions_manifest.csv"))
    for v, g in m.groupby("version"):
        low.append({"config": c, "version": v, "fotos": len(g), "conf media": round(g.conf_media.mean(), 3),
                    **{f"p<{t} (media)": round(g[f"frac_conf_lt_{t}"].mean(), 3) for t in P.CONF_THRESHOLDS}})
display(pd.DataFrame(low))

## 7. Hoja de contacto (4 etiquetadas de pliegues distintos con `_oof` + 4 sin etiquetar con el ensamble; semilla 42)

In [ ]:
legend = [(n, tuple(int(x) for x in rgb)) for n, rgb in zip(schema.names, palette)]
legend += [(f"sin anotar (CVAT: {c['name']})", tuple(c["rgb"])) for c in schema.source if c["name"] in schema.ignored]
cs = P.contact_sheet(OUT_ROOT, CONFIGS, photos, labelled, mask_rgb, legend, f"{OUT_ROOT}/hoja_de_contacto", seed=42)
print(cs["seleccion"])
from IPython.display import Image as IPImage
display(IPImage(str(cs["png"])))

## 8. Qué descargar

Por config, en `predictions/<config>/`: `masks_rgb.zip` y `masks_index.zip` (las 961 del ensamble), `oof.zip` (las 66 fuera de pliegue: `masks_index_oof`, `masks_rgb_oof`, `conf_oof`), `predictions_manifest.csv`, `models.json` y `verificacion_oof.json`. Además `predictions/hoja_de_contacto.png/.pdf/.csv`. La carpeta `conf/` (confianza a 1/4) queda en Drive; no va en los zips.